# 🤟 Sign Language Recognition — Training Notebook
**Pipeline:** Video MP4 → MediaPipe (2 tay) → BiLSTM → Nhãn từ

**Cấu trúc dataset trên Drive:**
```
MyDrive/NLP_DATASET/
  BAN/          ← train videos
  CAM_DIEC/
  DI/ GI/ HOC/ HOM_TRUOC/ KHONG/ THICH/ TOI/
  VALIDATION/
    BAN/ CAM_DIEC/ ...
  TEST/
    BAN/ CAM_DIEC/ ...
```

**Trước khi chạy:** `Runtime` → `Change runtime type` → **GPU (T4)**

## 1️⃣ Cài đặt thư viện

In [ ]:
!pip install mediapipe opencv-python torch torchvision tqdm scikit-learn matplotlib Pillow imageio imageio-ffmpeg -q
print('✅ Cài đặt xong!')

## 2️⃣ Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Chỉnh đường dẫn ở đây nếu cần ──
DATASET_ROOT = '/content/drive/MyDrive/NLP_DATASET'

TRAIN_DIR  = DATASET_ROOT                               # BAN/, CAM_DIEC/... nằm thẳng ở đây
VAL_DIR    = os.path.join(DATASET_ROOT, 'VALIDATION')
TEST_DIR   = os.path.join(DATASET_ROOT, 'TEST')
CACHE_DIR  = os.path.join(DATASET_ROOT, '_cache')
OUTPUT_DIR = os.path.join(DATASET_ROOT, '_output')

os.makedirs(os.path.join(CACHE_DIR, 'train'), exist_ok=True)
os.makedirs(os.path.join(CACHE_DIR, 'val'),   exist_ok=True)
os.makedirs(os.path.join(CACHE_DIR, 'test'),  exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'✅ Drive mounted')
print(f'   TRAIN_DIR  : {TRAIN_DIR}')
print(f'   VAL_DIR    : {VAL_DIR}')
print(f'   TEST_DIR   : {TEST_DIR}')
print(f'   OUTPUT_DIR : {OUTPUT_DIR}')
print(f'   Classes ({len(CLASSES)}): {CLASSES}')

## 3️⃣ Định nghĩa extractor (MediaPipe 2 tay)

In [ ]:
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm
import mediapipe as mp

FEATURE_DIM = 126   # 21 keypoints × 3 (x,y,z) × 2 tay
HAND_DIM    = 63
SEQ_LENGTH  = 35

CLASSES = [
    'BAN', 'CAM_DIEC', 'DI', 'GI', 'HOC',
    'HOM_TRUOC', 'KHONG', 'THICH', 'TOI'
]


class AdaptiveKeyframeExtractor:
    def __init__(self, target_frames=SEQ_LENGTH):
        self.target_frames = target_frames

    def extract_frame_indices(self, video_path):
        cap   = cv2.VideoCapture(video_path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
        if total <= 0:
            return []
        if total >= self.target_frames:
            return np.linspace(0, total - 1, self.target_frames, dtype=int).tolist()
        else:
            indices = list(range(total))
            while len(indices) < self.target_frames:
                indices.append(indices[-1])
            return indices


class HandKeypointExtractor:
    def __init__(self):
        mp_hands = mp.solutions.hands
        self.hands = mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=2,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
        )
        self.feature_dim = FEATURE_DIM

    def _landmarks_to_array(self, hand_landmarks):
        kps = [[lm.x, lm.y, lm.z] for lm in hand_landmarks.landmark]
        return np.array(kps, dtype=np.float32).flatten()

    def extract(self, frame):
        left_feat  = np.zeros(HAND_DIM, dtype=np.float32)
        right_feat = np.zeros(HAND_DIM, dtype=np.float32)
        try:
            frame_rgb  = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_results = self.hands.process(frame_rgb)
            if mp_results.multi_hand_landmarks:
                for landmarks, handedness in zip(
                    mp_results.multi_hand_landmarks,
                    mp_results.multi_handedness,
                ):
                    label = handedness.classification[0].label
                    feat  = self._landmarks_to_array(landmarks)
                    if label == 'Left':
                        left_feat  = feat
                    else:
                        right_feat = feat
        except Exception as e:
            print(f'[ExtractError] {e}')
        return np.concatenate([left_feat, right_feat])

    def __del__(self):
        if hasattr(self, 'hands'):
            self.hands.close()


class VideoFeatureCache:
    def __init__(self, cache_dir):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)

    def _path(self, video_path):
        return self.cache_dir / (Path(video_path).stem + '.npy')

    def exists(self, video_path):
        return self._path(video_path).exists()

    def save(self, video_path, arr):
        np.save(str(self._path(video_path)), arr)

    def load(self, video_path):
        p = self._path(video_path)
        if not p.exists():
            return None
        return np.load(str(p))


def list_videos(root, classes):
    vids, labels, counts = [], [], {}
    for ci, cname in enumerate(classes):
        p = Path(root) / cname
        if not p.exists():
            print(f'[Cảnh báo] Không tìm thấy: {p}')
            counts[cname] = 0
            continue
        files = list(p.glob('*.mp4')) + list(p.glob('*.avi')) + list(p.glob('*.MOV'))
        vids.extend([str(f) for f in files])
        labels.extend([ci] * len(files))
        counts[cname] = len(files)
    return vids, labels, counts


def precompute_features(videos, extractor, cache, seq_length=SEQ_LENGTH):
    to_process = [v for v in videos if not cache.exists(v)]
    if not to_process:
        print('Tất cả features đã được cache!')
        return
    print(f'Đang xử lý {len(to_process)} videos...')
    kf = AdaptiveKeyframeExtractor(target_frames=seq_length)
    for video_path in tqdm(to_process, desc='Extracting'):
        try:
            indices = kf.extract_frame_indices(video_path)
            cap     = cv2.VideoCapture(video_path)
            feats   = []
            for idx in indices:
                cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
                ok, frame = cap.read()
                feats.append(extractor.extract(frame) if ok else np.zeros(FEATURE_DIM, dtype=np.float32))
            cap.release()
            while len(feats) < seq_length:
                feats.append(feats[-1].copy() if feats else np.zeros(FEATURE_DIM, dtype=np.float32))
            cache.save(video_path, np.stack(feats[:seq_length]).astype(np.float32))
        except Exception as e:
            print(f'[Lỗi] {video_path}: {e}')
            cache.save(video_path, np.zeros((seq_length, FEATURE_DIM), dtype=np.float32))
    print('✅ Trích xuất features hoàn tất.')


print('✅ Extractor định nghĩa xong!')

## 4️⃣ Trích xuất features từ video

In [ ]:
print('Loading videos...')
train_vids, train_labels, train_counts = list_videos(TRAIN_DIR, CLASSES)
val_vids,   val_labels,   val_counts   = list_videos(VAL_DIR,   CLASSES)
test_vids,  test_labels,  test_counts  = list_videos(TEST_DIR,  CLASSES)

print('Train:', train_counts)
print('Val  :', val_counts)
print('Test :', test_counts)
print(f'Tổng: {len(train_vids)} train | {len(val_vids)} val | {len(test_vids)} test')

In [ ]:
extractor   = HandKeypointExtractor()
train_cache = VideoFeatureCache(os.path.join(CACHE_DIR, 'train'))
val_cache   = VideoFeatureCache(os.path.join(CACHE_DIR, 'val'))
test_cache  = VideoFeatureCache(os.path.join(CACHE_DIR, 'test'))

print('\n--- Trích xuất Train ---')
precompute_features(train_vids, extractor, train_cache)

print('\n--- Trích xuất Val ---')
precompute_features(val_vids, extractor, val_cache)

print('\n--- Trích xuất Test ---')
precompute_features(test_vids, extractor, test_cache)

del extractor
print('\n✅ Xong! Features đã lưu vào Drive.')

## 5️⃣ Định nghĩa Model & Dataset

In [ ]:
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
import matplotlib.pyplot as plt

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')


class HandDataset(Dataset):
    def __init__(self, videos, labels, cache, seq_length=SEQ_LENGTH, augment=False):
        self.videos     = videos
        self.labels     = labels
        self.cache      = cache
        self.seq_length = seq_length
        self.augment    = augment

    def __len__(self):
        return len(self.videos)

    def _mask_keypoints(self, x, prob=0.15):
        x = x.copy()
        for t in range(x.shape[0]):
            for offset in [0, 63]:
                frame = x[t, offset:offset+63].reshape(21, 3)
                for kp in range(21):
                    if random.random() < prob:
                        frame[kp] = 0.0
                x[t, offset:offset+63] = frame.flatten()
        return x

    def _temporal_shift(self, x):
        L     = x.shape[0]
        shift = random.randint(-L//5, L//5)
        if shift > 0:
            x = np.concatenate([np.repeat(x[:1], shift, axis=0), x[:-shift]], axis=0)
        elif shift < 0:
            x = np.concatenate([x[-shift:], np.repeat(x[-1:], -shift, axis=0)], axis=0)
        return x

    def __getitem__(self, idx):
        x = self.cache.load(self.videos[idx])
        if x is None:
            x = np.zeros((self.seq_length, FEATURE_DIM), dtype=np.float32)
        if self.augment:
            if random.random() < 0.5:
                x = self._mask_keypoints(x, prob=random.uniform(0.1, 0.2))
            if random.random() < 0.5:
                x = self._temporal_shift(x)
            if random.random() < 0.5:
                x = x + np.random.normal(0, 0.02, x.shape).astype(np.float32)
        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(int(self.labels[idx]), dtype=torch.long),
        )


class BiLSTMClassifier(nn.Module):
    def __init__(self, feature_dim=FEATURE_DIM, hidden=128, num_layers=2,
                 num_classes=10, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=feature_dim, hidden_size=hidden,
            num_layers=num_layers, batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout    = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(hidden * 2)
        self.classifier = nn.Linear(hidden * 2, num_classes)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        h = torch.cat([h[-2], h[-1]], dim=-1)
        h = self.dropout(h)
        h = self.layer_norm(h)
        return self.classifier(h)


class EarlyStopping:
    def __init__(self, patience=20):
        self.patience = patience
        self.best     = float('inf')
        self.counter  = 0

    def __call__(self, val_loss):
        if val_loss < self.best:
            self.best    = val_loss
            self.counter = 0
        else:
            self.counter += 1
        return self.counter >= self.patience


print('✅ Model & Dataset định nghĩa xong!')

## 6️⃣ Train model

In [ ]:
# ── Hyperparameters — chỉnh ở đây nếu cần ──
HIDDEN_SIZE  = 128
NUM_LAYERS   = 2
DROPOUT      = 0.3
LR           = 0.001
BATCH_SIZE   = 32
EPOCHS       = 100

# ── Datasets & Loaders ──
train_ds = HandDataset(train_vids, train_labels, train_cache, augment=True)
val_ds   = HandDataset(val_vids,   val_labels,   val_cache,   augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ── Model ──
model = BiLSTMClassifier(
    feature_dim=FEATURE_DIM,
    hidden=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_classes=len(CLASSES),
    dropout=DROPOUT,
).to(device)

optimizer  = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
criterion  = nn.CrossEntropyLoss(label_smoothing=0.1)
early_stop = EarlyStopping(patience=20)
best_val_loss = float('inf')

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, 'best_model.pth')

print(f'Bắt đầu train {EPOCHS} epochs...')
print('='*55)

for epoch in range(EPOCHS):
    # ── Train ──
    model.train()
    total_loss = correct = total = 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out  = model(X)
        loss = criterion(out, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        correct    += (out.argmax(1) == y).sum().item()
        total      += y.size(0)

    train_loss = total_loss / len(train_loader)
    train_acc  = correct / total

    # ── Validate ──
    model.eval()
    val_loss = v_correct = v_total = 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)
            out  = model(X)
            val_loss   += criterion(out, y).item()
            v_correct  += (out.argmax(1) == y).sum().item()
            v_total    += y.size(0)

    val_loss /= len(val_loader)
    val_acc   = v_correct / v_total

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f'Epoch {epoch+1:3d} | Train {train_loss:.4f}/{train_acc*100:.1f}% | Val {val_loss:.4f}/{val_acc*100:.1f}%', end='')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(' ← best', end='')

    print()

    if early_stop(val_loss):
        print(f'\nEarly stopping tại epoch {epoch+1}')
        break

print(f'\n✅ Train xong! Model lưu tại: {MODEL_SAVE_PATH}')

## 7️⃣ Vẽ biểu đồ training

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'],   label='Val Loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot([a*100 for a in history['train_acc']], label='Train Acc')
axes[1].plot([a*100 for a in history['val_acc']],   label='Val Acc')
axes[1].set_title('Accuracy (%)')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_history.png'), dpi=150)
plt.show()
print('✅ Biểu đồ đã lưu vào Drive.')

## 8️⃣ Đánh giá trên Test set

In [ ]:
# Load best model
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
model.eval()

test_ds     = HandDataset(test_vids, test_labels, test_cache, augment=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

all_preds, all_labels_list = [], []
with torch.no_grad():
    for X, y in test_loader:
        out = model(X.to(device))
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels_list.extend(y.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels_list)

print('='*55)
print('TEST RESULTS')
print('='*55)
print(classification_report(all_labels, all_preds, target_names=CLASSES))
print(f'Balanced Accuracy: {balanced_accuracy_score(all_labels, all_preds)*100:.2f}%')

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(CLASSES)))
ax.set_yticks(range(len(CLASSES)))
ax.set_xticklabels(CLASSES, rotation=45, ha='right')
ax.set_yticklabels(CLASSES)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()
print('✅ Confusion matrix đã lưu vào Drive.')

## 9️⃣ Tải model về máy
Sau khi train xong, chạy cell này để tải `best_model.pth` về máy tính, đặt vào thư mục `output/` cùng với các file `.py`.

In [ ]:
from google.colab import files
files.download(MODEL_SAVE_PATH)
print('✅ Đang tải best_model.pth về máy...')

In [ ]:
Run this to extract metadata Json and state_dict from the best model.

In [ ]:
import json
from pathlib import Path
from datetime import datetime
import torch

EXPORT_DIR = Path(OUTPUT_DIR)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

STATE_DICT_PATH = EXPORT_DIR / "vsl_bilstm_state_dict.pth"
METADATA_PATH = EXPORT_DIR / "vsl_bilstm_metadata.json"
MODEL_CLASS_PATH = EXPORT_DIR / "model.py"

torch.save(model.state_dict(), STATE_DICT_PATH)

best_val_acc = max(history.get("val_acc", [0])) if history.get("val_acc") else 0.0
best_val_loss = min(history.get("val_loss", [999.0])) if history.get("val_loss") else None

metadata = {
    "model_name": "vsl_bilstm_classifier",
    "model_family": "bilstm",
    "export_format": "pytorch",
    "version": "v1",
    "is_state_dict": True,
    "has_model_class": True,

    "labels": CLASSES,

    "modality": "cv",
    "pipeline_stage": "classifier",
    "language": "vi",
    "locale": "vi-VN",

    "input_spec": {
        "shape": [1, SEQ_LENGTH, FEATURE_DIM],
        "input_dim": SEQ_LENGTH * FEATURE_DIM,
        "sequence_length": SEQ_LENGTH,
        "feature_dim": FEATURE_DIM,
        "modality": "cv",
        "preprocess_profile": "wrist_center"
    },

    "architecture": {
        "feature_dim": FEATURE_DIM,
        "hidden": HIDDEN_SIZE,
        "num_layers": NUM_LAYERS,
        "num_classes": len(CLASSES),
        "dropout": DROPOUT,
        "bidirectional": True
    },

    "hyperparameters": {
        "learning_rate": LR,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "optimizer": "AdamW",
        "loss": "CrossEntropyLoss",
        "label_smoothing": 0.1,
        "weight_decay": 0.01
    },

    "precision": float(best_val_acc),
    "recall": float(best_val_acc),
    "f1": float(best_val_acc),

    "training_summary": {
        "best_val_accuracy": float(best_val_acc),
        "best_val_loss": float(best_val_loss) if best_val_loss is not None else None
    },

    "notes": "Vietnamese Sign Language BiLSTM classifier over MediaPipe 2-hand keypoint sequences.",
    "created_at": datetime.utcnow().isoformat() + "Z"
}

METADATA_PATH.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Saved:", STATE_DICT_PATH)
print("Saved:", METADATA_PATH)


In [ ]:
from pathlib import Path

model_class_code = f"""
import torch
import torch.nn as nn

FEATURE_DIM = {FEATURE_DIM}
HIDDEN_SIZE = {HIDDEN_SIZE}
NUM_LAYERS = {NUM_LAYERS}
NUM_CLASSES = {len(CLASSES)}
DROPOUT = {DROPOUT}

class BiLSTMClassifier(nn.Module):
    def __init__(
        self,
        feature_dim=FEATURE_DIM,
        hidden=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        num_classes=NUM_CLASSES,
        dropout=DROPOUT
    ):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=feature_dim,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(hidden * 2)
        self.classifier = nn.Linear(hidden * 2, num_classes)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        h = torch.cat([h[-2], h[-1]], dim=-1)
        h = self.dropout(h)
        h = self.layer_norm(h)
        return self.classifier(h)
""".strip()

MODEL_CLASS_PATH.write_text(model_class_code, encoding="utf-8")
print("Saved:", MODEL_CLASS_PATH)
